# Integrated Colab Tutorial: From Data Source to Defensible Data Product

**Weeks 1–4 | Data gathering, cleaning, structures, and storage**

This notebook follows one workplace case from beginning to end. You will evaluate potential data sources, clean a messy service-request extract, verify the result, and then compare several ways to represent and store the same information.

**Audience:** graduate data science students, including learners without a computer-science background  
**Platform:** Google Colab free tier  
**Estimated time:** 150–180 minutes across four modules, plus the applied exercise  
**Dependencies:** Python standard library and `pandas`, already available in Colab

> The goal is not merely to produce a table. The goal is to produce data that another person can understand, check, and use responsibly.


## Learning objectives

By the end of this tutorial, you should be able to:

1. **Evaluate** a workplace data source for purpose, access, ownership, sensitivity, and limitations.
2. **Translate** a data request into a small, testable processing specification.
3. **Clean and validate** a messy CSV extract without silently hiding quality problems.
4. **Retrieve and process** a bounded sample from an official public-data portal.
5. **Compare** dictionaries, flat tables, normalized tables, nested JSON, and event records.
6. **Choose and justify** a representation based on the intended use.
7. **Preserve evidence** of provenance, validation, and successful save/reload checks.


## The workplace case

A campus facilities office wants a reliable weekly view of maintenance requests. Staff currently exchange spreadsheets, while a future application may need structured records and status history.

We will produce five deliverables:

- a source register documenting where the data could come from;
- a cleaned, analysis-ready request table;
- alternative representations for application and reporting needs;
- a small evidence package containing saved files and verification results;
- a cleaned public-data sample with its query boundary and limitations.

The facilities records are synthetic. Week 4 then applies the workflow to a bounded sample from an official public-data portal.


## Workflow map

```text
Purpose and source
        ↓
Raw extract and quality inspection
        ↓
Cleaning, validation, and provenance
        ↓
Dictionaries and analysis table
        ↓
Relational tables, nested JSON, and events
        ↓
Storage decision and verified artifacts
        ↓
Bounded public-data retrieval and comparison
```

Run the notebook from top to bottom. Each part produces an object used later.


## 0. Set up the notebook once


In [ ]:
from copy import deepcopy
from io import StringIO
from pathlib import Path
import json
import sqlite3

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
OUTPUT_DIR = Path("weeks1_3_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("pandas version:", pd.__version__)
print("output folder:", OUTPUT_DIR.resolve())


# Part 1 — Begin with purpose, source, and risk

A technically convenient source is not automatically an appropriate source. Before collecting data, document its owner, access method, refresh pattern, sensitivity, intended use, and limitations.

This is a small **source register**. In a real organization, a data steward would confirm these entries with the source owners.


In [ ]:
source_register = pd.DataFrame(
    [
        {
            "source_id": "work_orders",
            "source_name": "Facilities work-order export",
            "owner": "Facilities Operations",
            "access_method": "Authorized CSV export",
            "refresh": "Daily",
            "sensitivity": "Internal; descriptions may contain personal details",
            "intended_use": "Weekly workload and resolution reporting",
            "known_limit": "Manual entry creates inconsistent categories",
        },
        {
            "source_id": "buildings",
            "source_name": "Campus building directory",
            "owner": "Campus Planning",
            "access_method": "Approved reference file",
            "refresh": "Quarterly",
            "sensitivity": "Low",
            "intended_use": "Translate building codes into names",
            "known_limit": "Renamed buildings may appear late",
        },
        {
            "source_id": "assignments",
            "source_name": "Technician assignment roster",
            "owner": "Facilities Operations",
            "access_method": "Restricted database view",
            "refresh": "Near real time",
            "sensitivity": "Employee operational data",
            "intended_use": "Coordinate work, not evaluate employees",
            "known_limit": "Assignments change during the day",
        },
    ]
)

display(source_register)


### Check that the source documentation is usable


In [ ]:
required_source_fields = {
    "source_id", "owner", "access_method", "sensitivity",
    "intended_use", "known_limit"
}

source_checks = {
    "required columns present": required_source_fields.issubset(source_register.columns),
    "source IDs are unique": source_register["source_id"].is_unique,
    "owners are documented": source_register["owner"].notna().all(),
    "limitations are documented": source_register["known_limit"].notna().all(),
}

display(pd.Series(source_checks, name="passed").to_frame())
assert all(source_checks.values())


### Stewardship decision

For this tutorial, we will use a synthetic copy of the work-order export. That supports learning without exposing real descriptions, locations, or employee information.

A useful source decision answers:

- What question are we permitted to answer?
- Who owns the source and who may access it?
- Which fields are necessary?
- What could be misunderstood or misused?
- How will we record transformations and quality problems?


# Part 2 — Turn a messy extract into trustworthy evidence


## 2.1 Write the processing specification before the code

Treat an AI prompt—or any coding request—as a draft specification:

> Load the facilities CSV without changing the raw values. Preserve one output row per request. Trim and standardize categorical text; parse dates and costs; retain failed parses as missing values; add a review flag; verify identifiers, row counts, allowed status values, and required columns; then save and reload the cleaned file.

This statement defines inputs, transformations, expected outputs, and checks. It is more useful than “clean this data.”


## 2.2 Load the raw extract


In [ ]:
raw_csv = '''request_id,opened_at,building_code,agency_code,category,priority,status,estimated_cost
 WO-1001 ,2026-09-01, LIB , FAC , hvac ,high, open ,1250.00
WO-1002,09/02/2026,SCI,FAC,Electrical, medium,In Progress,825
WO-1003,2026-09-03,,FAC,plumbing,LOW,closed,unknown
WO-1004,not recorded,ADM,SEC,Access Control,high,OPEN,450
WO-1005,2026-09-05,LIB,IT,network,Medium,in_progress,300.50
WO-1006,2026-09-06,SCI,FAC,HVAC,high,Closed,2100
WO-1007,2026-09-07,ADM,FAC,plumbing, low,pending,
'''

raw_requests = pd.read_csv(StringIO(raw_csv), dtype=str)
display(raw_requests)


## 2.3 Inspect before transforming

Inspection distinguishes what we observed from what we assume. Check shape, data types, missing values, identifier duplication, and inconsistent categories before cleaning anything.


In [ ]:
print("shape:", raw_requests.shape)
print("\ndata types:")
display(raw_requests.dtypes.to_frame("dtype"))

print("missing values:")
display(raw_requests.isna().sum().to_frame("missing_count"))

for column in ["category", "priority", "status"]:
    print(f"{column}: {raw_requests[column].dropna().unique().tolist()}")


## 2.4 Clean with explicit rules

The code below keeps the raw table unchanged and creates a new `clean_requests` table. Standardization rules are visible, and conversion failures are flagged instead of silently discarded.


In [ ]:
clean_requests = raw_requests.copy()

# Trim surrounding spaces from every text column.
for column in clean_requests.columns:
    clean_requests[column] = clean_requests[column].str.strip()

# Standardize identifiers and categories.
for column in ["request_id", "building_code", "agency_code"]:
    clean_requests[column] = clean_requests[column].str.upper()

clean_requests["category"] = clean_requests["category"].str.title()
clean_requests["priority"] = clean_requests["priority"].str.lower()

status_map = {
    "open": "OPEN",
    "in progress": "IN_PROGRESS",
    "in_progress": "IN_PROGRESS",
    "closed": "CLOSED",
    "pending": "PENDING",
}
clean_requests["status"] = (
    clean_requests["status"].str.lower().map(status_map)
)

# Parse values. Invalid values become missing and remain visible in review flags.
clean_requests["opened_at"] = pd.to_datetime(
    clean_requests["opened_at"], errors="coerce", format="mixed"
)
clean_requests["estimated_cost"] = pd.to_numeric(
    clean_requests["estimated_cost"], errors="coerce"
)

def quality_issue(row):
    issues = []
    if pd.isna(row["building_code"]):
        issues.append("missing building")
    if pd.isna(row["opened_at"]):
        issues.append("invalid date")
    if pd.isna(row["estimated_cost"]):
        issues.append("missing or invalid cost")
    if pd.isna(row["status"]):
        issues.append("unrecognized status")
    return "; ".join(issues)

clean_requests["quality_issue"] = clean_requests.apply(quality_issue, axis=1)
clean_requests["needs_review"] = clean_requests["quality_issue"].ne("")

display(clean_requests)


> **Version note:** `format="mixed"` works in the version of pandas used by current Colab runtimes. If an older environment rejects it, remove that argument and parse the formats separately.

## 2.5 Validate the output

Validation is executable evidence. A notebook that runs without an error is not necessarily correct.


In [ ]:
expected_columns = {
    "request_id", "opened_at", "building_code", "agency_code",
    "category", "priority", "status", "estimated_cost",
    "quality_issue", "needs_review"
}
allowed_statuses = {"OPEN", "IN_PROGRESS", "CLOSED", "PENDING"}

validation_checks = {
    "row count preserved": len(clean_requests) == len(raw_requests),
    "required columns present": expected_columns.issubset(clean_requests.columns),
    "request IDs complete": clean_requests["request_id"].notna().all(),
    "request IDs unique": clean_requests["request_id"].is_unique,
    "status values allowed": set(clean_requests["status"].dropna()).issubset(allowed_statuses),
    "quality problems flagged": clean_requests["needs_review"].sum() == 3,
}

validation_evidence = pd.Series(validation_checks, name="passed").to_frame()
display(validation_evidence)
assert all(validation_checks.values())


## 2.6 Summarize only after validation


In [ ]:
workload_summary = (
    clean_requests
    .groupby(["agency_code", "status"], dropna=False)
    .agg(
        request_count=("request_id", "count"),
        known_estimated_cost=("estimated_cost", "sum"),
    )
    .reset_index()
    .sort_values(["agency_code", "status"])
)

display(workload_summary)
print(
    "Caution: cost totals exclude missing or invalid values, so they are not "
    "complete budget figures."
)


## 2.7 Record basic provenance

Provenance answers: Where did the data come from, what happened to it, and what evidence was produced? This small log is not an enterprise lineage system, but it establishes the habit.


In [ ]:
provenance_log = pd.DataFrame(
    [
        {
            "stage": "source",
            "input": "synthetic work-order CSV",
            "action": "loaded all fields as text",
            "rows": len(raw_requests),
            "evidence": "raw_requests shape and inspection",
        },
        {
            "stage": "clean",
            "input": "raw_requests",
            "action": "trimmed text; standardized categories; parsed date and cost",
            "rows": len(clean_requests),
            "evidence": f"{clean_requests['needs_review'].sum()} rows flagged for review",
        },
        {
            "stage": "validate",
            "input": "clean_requests",
            "action": "checked schema, row count, IDs, status domain, and flags",
            "rows": len(clean_requests),
            "evidence": f"{sum(validation_checks.values())}/{len(validation_checks)} checks passed",
        },
    ]
)

display(provenance_log)


# Part 3 — Represent the same workplace information in different ways

There is no universally best data structure. The right choice depends on whether the task is editing one record, analyzing many records, enforcing relationships, exchanging hierarchical data, or preserving change history.


## 3.1 Dictionary: convenient for one record


In [ ]:
request_record = clean_requests.iloc[0].to_dict()

print("request:", request_record["request_id"])
print("building:", request_record.get("building_code", "UNKNOWN"))

revised_record = deepcopy(request_record)
revised_record["status"] = "IN_PROGRESS"

print("original status:", request_record["status"])
print("revised status:", revised_record["status"])


A dictionary maps field names to values and is easy to read or update. However, a collection of dictionaries does not automatically enforce consistent fields, data types, or relationships.

## 3.2 DataFrame: convenient for analysis

The cleaned DataFrame supports filtering, grouping, and column-wise validation. Its weakness is duplication: agency and building descriptions would be repeated on many request rows.


In [ ]:
high_priority_requests = clean_requests.loc[
    clean_requests["priority"].eq("high"),
    ["request_id", "building_code", "category", "status", "estimated_cost"]
]

display(high_priority_requests)


## 3.3 Normalized tables: convenient for controlled relationships

Separate stable entities from recurring transactions:

- `agencies`: one row per responsible unit;
- `buildings`: one row per campus building;
- `requests`: one row per work request, referring to codes in the other tables.

Those references behave like **foreign keys**. They reduce repeated descriptions and make ownership clearer.


In [ ]:
agencies = pd.DataFrame(
    [
        {"agency_code": "FAC", "agency_name": "Facilities Operations"},
        {"agency_code": "SEC", "agency_name": "Campus Security"},
        {"agency_code": "IT", "agency_name": "Information Technology"},
    ]
)

buildings = pd.DataFrame(
    [
        {"building_code": "LIB", "building_name": "Library"},
        {"building_code": "SCI", "building_name": "Science Center"},
        {"building_code": "ADM", "building_name": "Administration Building"},
    ]
)

requests = clean_requests[
    [
        "request_id", "opened_at", "building_code", "agency_code",
        "category", "priority", "status", "estimated_cost", "needs_review"
    ]
].copy()

request_view = (
    requests
    .merge(agencies, on="agency_code", how="left", validate="many_to_one")
    .merge(buildings, on="building_code", how="left", validate="many_to_one")
)

display(request_view.head())


### Check referential integrity rather than assuming it


In [ ]:
relationship_checks = {
    "agency codes resolve": request_view["agency_name"].notna().all(),
    "known building codes resolve": request_view.loc[
        request_view["building_code"].notna(), "building_name"
    ].notna().all(),
    "request rows preserved by joins": len(request_view) == len(requests),
}

display(pd.Series(relationship_checks, name="passed").to_frame())
assert all(relationship_checks.values())


## 3.4 Nested JSON: convenient for exchange and application responses

JSON can keep related information together in one hierarchical object. That is useful for APIs and documents, although organization-wide updates and cross-record analysis can be harder than in relational tables.


In [ ]:
first = request_view.iloc[0]
request_document = {
    "request_id": first["request_id"],
    "opened_at": first["opened_at"].isoformat(),
    "category": first["category"],
    "priority": first["priority"],
    "status": first["status"],
    "agency": {
        "code": first["agency_code"],
        "name": first["agency_name"],
    },
    "location": {
        "code": first["building_code"],
        "name": first["building_name"],
    },
    "estimated_cost": first["estimated_cost"],
}

print(json.dumps(request_document, indent=2))


In [ ]:
flattened_document = pd.json_normalize(request_document, sep="_")
display(flattened_document)


## 3.5 Events: preserve what changed and when

A current-state table tells us the latest status. An event log preserves transitions. This is valuable for auditability and process analysis.

**Optional advanced extension:** distributed systems may deliver an event more than once. An `event_id` lets us remove duplicates before reconstructing current state; this is a small demonstration of idempotent processing.


In [ ]:
events = pd.DataFrame(
    [
        {"event_id": "E-001", "request_id": "WO-1001", "event_time": "2026-09-01 08:30", "new_status": "OPEN"},
        {"event_id": "E-002", "request_id": "WO-1001", "event_time": "2026-09-01 10:15", "new_status": "IN_PROGRESS"},
        {"event_id": "E-003", "request_id": "WO-1001", "event_time": "2026-09-02 16:40", "new_status": "CLOSED"},
        {"event_id": "E-004", "request_id": "WO-1002", "event_time": "2026-09-02 09:10", "new_status": "OPEN"},
        {"event_id": "E-005", "request_id": "WO-1002", "event_time": "2026-09-03 11:20", "new_status": "IN_PROGRESS"},
    ]
)
events["event_time"] = pd.to_datetime(events["event_time"])

# Simulate a duplicate delivery, then make processing safe by event ID.
delivered_events = pd.concat([events, events.iloc[[1]]], ignore_index=True)
unique_events = delivered_events.drop_duplicates("event_id")

current_status_from_events = (
    unique_events
    .sort_values("event_time")
    .groupby("request_id", as_index=False)
    .tail(1)[["request_id", "new_status", "event_time"]]
    .rename(columns={"new_status": "current_status"})
)

print("delivered rows:", len(delivered_events))
print("unique events:", len(unique_events))
display(current_status_from_events)


## 3.6 SQLite and SQL: query related tables locally

This is a small SQL preview, not a database-administration exercise. SQLite is built into Python and runs inside Colab without a server.


In [ ]:
database_path = OUTPUT_DIR / "facilities_demo.sqlite"
connection = sqlite3.connect(database_path)

agencies.to_sql("agencies", connection, if_exists="replace", index=False)
buildings.to_sql("buildings", connection, if_exists="replace", index=False)
requests.to_sql("requests", connection, if_exists="replace", index=False)

sql = '''
SELECT
    a.agency_name,
    r.status,
    COUNT(*) AS request_count,
    ROUND(SUM(r.estimated_cost), 2) AS known_estimated_cost
FROM requests AS r
LEFT JOIN agencies AS a
    ON r.agency_code = a.agency_code
GROUP BY a.agency_name, r.status
ORDER BY a.agency_name, r.status;
'''

sql_summary = pd.read_sql_query(sql, connection)
display(sql_summary)
connection.close()


# Part 4 — Choose a representation and preserve evidence


In [ ]:
model_comparison = pd.DataFrame(
    [
        {
            "representation": "Dictionary",
            "best_for": "One record or configuration object",
            "main_strength": "Simple named fields",
            "main_tradeoff": "Weak consistency across many records",
        },
        {
            "representation": "DataFrame / flat table",
            "best_for": "Cleaning, analysis, and reporting",
            "main_strength": "Strong column operations",
            "main_tradeoff": "Repeated descriptive values",
        },
        {
            "representation": "Normalized relational tables",
            "best_for": "Shared operational records and controlled relationships",
            "main_strength": "Reduced duplication and enforceable links",
            "main_tradeoff": "Requires joins and schema planning",
        },
        {
            "representation": "Nested JSON document",
            "best_for": "APIs and self-contained hierarchical records",
            "main_strength": "Related data travels together",
            "main_tradeoff": "Cross-document updates and analysis are harder",
        },
        {
            "representation": "Event log",
            "best_for": "History, audit, and process reconstruction",
            "main_strength": "Preserves change over time",
            "main_tradeoff": "Current state must be derived",
        },
    ]
)

display(model_comparison)


## Decision practice

Choose based on the work requirement, not on which technology sounds most advanced:

1. Weekly workload chart → cleaned DataFrame or SQL query.
2. Operational system with shared agency and building records → relational tables.
3. API response containing one complete request → nested JSON.
4. Audit of how long requests remained open → event log.
5. One temporary record inside Python → dictionary.

Real systems often use several representations together. Stewardship requires documenting which copy is authoritative and how the representations remain consistent.


## Save, reload, and verify the evidence package


In [ ]:
source_path = OUTPUT_DIR / "source_register.csv"
clean_path = OUTPUT_DIR / "clean_requests.csv"
provenance_path = OUTPUT_DIR / "provenance_log.csv"
document_path = OUTPUT_DIR / "request_example.json"
events_path = OUTPUT_DIR / "status_events.jsonl"

source_register.to_csv(source_path, index=False)
clean_requests.to_csv(clean_path, index=False)
provenance_log.to_csv(provenance_path, index=False)
document_path.write_text(json.dumps(request_document, indent=2), encoding="utf-8")
events.to_json(events_path, orient="records", lines=True, date_format="iso")

reloaded_clean = pd.read_csv(clean_path)
reloaded_document = json.loads(document_path.read_text(encoding="utf-8"))
reloaded_events = pd.read_json(events_path, lines=True)

artifact_checks = {
    "clean CSV exists": clean_path.exists(),
    "clean CSV row count preserved": len(reloaded_clean) == len(clean_requests),
    "JSON request ID preserved": reloaded_document["request_id"] == request_document["request_id"],
    "JSONL event count preserved": len(reloaded_events) == len(events),
    "SQLite database exists": database_path.exists(),
}

display(pd.Series(artifact_checks, name="passed").to_frame())
assert all(artifact_checks.values())
print("Saved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print("-", path)


# Part 5 — Week 4: Process a public open dataset

The synthetic facilities extract was useful because we controlled its problems. Real public data adds new complications: a remote source, changing categories, incomplete fields, unclear population boundaries, and documentation that must be read alongside the records.

We will retrieve a bounded sample from **NYC Open Data: 311 Service Requests from 2020 to Present**. The dataset is maintained by NYC OpenData and 311, updated daily, and contains service requests directed to city agencies.

- [Dataset page](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9)
- [Official API documentation](https://dev.socrata.com/foundry/data.cityofnewyork.us/erm2-nwe9)
- Dataset identifier: `erm2-nwe9`

We deliberately request only selected fields and at most 1,000 rows from a fixed historical week. This keeps the activity reproducible and appropriate for Colab's free tier. No API key is required for this small request.


## 5.1 Document the public source and query boundary

“Public” does not mean “perfect,” “complete,” or “appropriate for every use.” Record who publishes the data, what one row represents, which period was requested, which fields were excluded, and how the row limit affects interpretation.


In [ ]:
public_source_record = {
    "source_name": "NYC 311 Service Requests from 2020 to Present",
    "publisher": "NYC OpenData / NYC 311",
    "dataset_id": "erm2-nwe9",
    "unit_of_observation": "One service request directed to a city agency",
    "requested_period": "2025-01-01 through 2025-01-07",
    "maximum_rows": 1000,
    "selected_geography": "Borough and incident ZIP; exact coordinates excluded",
    "known_limit": "The limited sample is not the complete week and the source is updated daily",
}

display(pd.Series(public_source_record, name="value").to_frame())


## 5.2 Retrieve a small reproducible sample


In [ ]:
from urllib.parse import urlencode

public_api_base = "https://data.cityofnewyork.us/resource/erm2-nwe9.csv"
public_query = {
    "$limit": 1000,
    "$select": (
        "unique_key,created_date,closed_date,agency,complaint_type,"
        "descriptor,incident_zip,borough,status"
    ),
    "$where": (
        "created_date between '2025-01-01T00:00:00.000' "
        "and '2025-01-07T23:59:59.999'"
    ),
    "$order": "created_date, unique_key",
}
public_data_url = public_api_base + "?" + urlencode(public_query)

try:
    public_raw = pd.read_csv(
        public_data_url,
        dtype={"unique_key": "string", "incident_zip": "string"},
    )
except Exception as error:
    raise RuntimeError(
        "The public-data request failed. Confirm that the Colab runtime has "
        "internet access, then retry the cell."
    ) from error

print("rows retrieved:", len(public_raw))
print("columns retrieved:", len(public_raw.columns))
display(public_raw.head())


The URL is constructed from explicit query parameters. That makes the extraction boundary auditable: another analyst can see the date window, selected columns, ordering, and row limit.

## 5.3 Inspect the public records before cleaning


In [ ]:
print("shape:", public_raw.shape)
display(public_raw.dtypes.to_frame("dtype"))
display(public_raw.isna().sum().to_frame("missing_count"))

print("agencies represented:", public_raw["agency"].nunique(dropna=True))
print("complaint types represented:", public_raw["complaint_type"].nunique(dropna=True))
print("borough values:", sorted(public_raw["borough"].dropna().unique().tolist()))


## 5.4 Clean without inventing missing information

We standardize text, parse timestamps, and calculate resolution time when both timestamps are available. Missing closure times remain missing. A missing value should not automatically be interpreted as an error or an open request; consult the dataset documentation and business rules before making that claim.


In [ ]:
public_clean = public_raw.copy()
public_clean = public_clean.rename(columns={"unique_key": "request_id"})

text_columns = [
    "agency", "complaint_type", "descriptor", "incident_zip", "borough", "status"
]
for column in text_columns:
    public_clean[column] = public_clean[column].astype("string").str.strip()

public_clean["agency"] = public_clean["agency"].str.upper()
public_clean["borough"] = public_clean["borough"].str.upper()
public_clean["status"] = public_clean["status"].str.upper()
public_clean["created_date"] = pd.to_datetime(
    public_clean["created_date"], errors="coerce"
)
public_clean["closed_date"] = pd.to_datetime(
    public_clean["closed_date"], errors="coerce"
)

public_clean["resolution_hours"] = (
    public_clean["closed_date"] - public_clean["created_date"]
).dt.total_seconds() / 3600
public_clean["timing_issue"] = public_clean["resolution_hours"].lt(0).fillna(False)
public_clean["has_closed_timestamp"] = public_clean["closed_date"].notna()

display(public_clean.head())


## 5.5 Validate the extraction and transformation


In [ ]:
public_required_columns = {
    "request_id", "created_date", "closed_date", "agency", "complaint_type",
    "descriptor", "incident_zip", "borough", "status", "resolution_hours",
    "timing_issue", "has_closed_timestamp"
}

public_checks = {
    "records were retrieved": len(public_clean) > 0,
    "API row limit respected": len(public_clean) <= public_source_record["maximum_rows"],
    "required columns present": public_required_columns.issubset(public_clean.columns),
    "request IDs complete": public_clean["request_id"].notna().all(),
    "request IDs unique": public_clean["request_id"].is_unique,
    "created timestamps parsed": public_clean["created_date"].notna().all(),
    "negative durations are flagged": public_clean.loc[
        public_clean["resolution_hours"].lt(0), "timing_issue"
    ].all(),
}

display(pd.Series(public_checks, name="passed").to_frame())
assert all(public_checks.values())


## 5.6 Produce bounded descriptive summaries


In [ ]:
top_complaints = (
    public_clean
    .groupby(["agency", "complaint_type"], dropna=False)
    .size()
    .rename("request_count")
    .reset_index()
    .sort_values("request_count", ascending=False)
    .head(10)
)

borough_summary = (
    public_clean
    .groupby("borough", dropna=False)
    .agg(
        request_count=("request_id", "count"),
        records_with_close_time=("has_closed_timestamp", "sum"),
        median_resolution_hours=("resolution_hours", "median"),
    )
    .reset_index()
    .sort_values("request_count", ascending=False)
)

print("Top agency and complaint-type combinations in the retrieved sample:")
display(top_complaints)
print("Borough summary for the retrieved sample:")
display(borough_summary)


### Interpretation boundary

These summaries describe only the ordered, 1,000-row extract—not all NYC requests during the week. Request counts do not measure resident behavior, agency quality, neighborhood need, or service performance by themselves. Reporting should disclose the query, row limit, missing values, update timing, and relevant operational context.


## 5.7 Save, reload, and document the public-data artifact


In [ ]:
public_clean_path = OUTPUT_DIR / "nyc_311_public_sample_clean.csv"
public_metadata_path = OUTPUT_DIR / "nyc_311_public_sample_metadata.json"

public_clean.to_csv(public_clean_path, index=False)
public_metadata_path.write_text(
    json.dumps(
        {
            **public_source_record,
            "dataset_page": (
                "https://data.cityofnewyork.us/Social-Services/"
                "311-Service-Requests-from-2020-to-Present/erm2-nwe9"
            ),
            "api_base": public_api_base,
            "query_parameters": public_query,
            "retrieved_rows": len(public_clean),
            "validation_checks": {
                name: bool(result) for name, result in public_checks.items()
            },
        },
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

public_reloaded = pd.read_csv(public_clean_path, dtype={"request_id": "string"})
public_reload_checks = {
    "public CSV exists": public_clean_path.exists(),
    "metadata exists": public_metadata_path.exists(),
    "row count preserved": len(public_reloaded) == len(public_clean),
    "request IDs preserved": set(public_reloaded["request_id"]) == set(public_clean["request_id"]),
}

display(pd.Series(public_reload_checks, name="passed").to_frame())
assert all(public_reload_checks.values())


## Week 4 checkpoint

Submit or demonstrate:

1. the successfully executed retrieval, inspection, cleaning, and validation cells;
2. `nyc_311_public_sample_clean.csv` and its metadata JSON file;
3. one accurate observation supported by a displayed summary;
4. one limitation explaining why the sample cannot support a broader claim;
5. a comparison of one quality issue in the public data with one issue in the synthetic facilities data.

**AI policy:** Follow the course syllabus. AI may support instructor-authorized brainstorming or learning, but it must not complete a graded submission unless the instructor explicitly permits that use. Document any permitted assistance and how you verified it.


# Applied exercise: adapt the workflow to your workplace

Choose a small, non-sensitive dataset from your professional area—or create a synthetic one. Complete the following:

1. Add one entry to a source register, including owner, access, sensitivity, intended use, and limitation.
2. Create or load 8–20 messy records with at least three realistic quality problems.
3. Write a processing specification before using any instructor-permitted AI assistance. If the work is graded, follow the syllabus restriction on AI-generated code.
4. Preserve the raw data and create a separate cleaned table.
5. Add at least four validation checks and one review flag.
6. Represent one record as nested JSON **or** divide the table into two related tables.
7. Save, reload, and verify the principal artifact.
8. Write a 150–250 word decision note explaining:
   - why the data is appropriate for the stated purpose;
   - what remains uncertain or incomplete;
   - why your selected representation fits the intended use;
   - what permitted AI assistance you used and checked, or that you used none.

**Do not submit confidential, personally identifiable, regulated, or employer-owned data without authorization.** Synthetic data is acceptable and encouraged.


## Evaluation checklist (10 points)

| Criterion | Points |
|---|---:|
| Source, purpose, ownership, sensitivity, and limitation are documented | 2 |
| Cleaning rules address realistic problems without overwriting the raw data | 2 |
| Validation provides convincing evidence, including review flags | 2 |
| Representation choice is implemented and justified | 2 |
| Files reload successfully; limitations and AI use are documented | 2 |

## Key takeaways

- Begin with purpose and authority, not with code.
- Preserve raw evidence and make transformations explicit.
- Failed parsing is information; flag it rather than hiding it.
- A DataFrame, relational model, JSON document, and event log solve different problems.
- Verification, provenance, and limitations are part of the data product—not optional commentary.
